# Day 66 — Baseline Model: Retail Demand Predictor Mini-Project

This notebook covers Baseline Linear Regression Model, Training Pipeline Sanity Check, Evaluation Metrics (MAE, RMSE, $R^2$), Additive vs. Multiplicative Limitations, and Prediction Export

## Validation goals
- Build and train a baseline linear regression model using preprocessed training splits to establish an empirical performance floor
- Compute core regression metrics (MAE, RMSE, and $R^2$) on the test set to quantify baseline accuracy
- Export test predictions to `retail_demand_predictions.csv` for structured comparison against advanced models
- Analyze the theoretical limitations of additive linear models when handling multiplicative seasonal demand patterns

### Why Build a Baseline First?

In applied machine learning, you never jump straight to complex ensemble models or deep neural networks. You always build a simple, fast, interpretable baseline model first:
- Sanity Check: It proves your end-to-end data pipeline works without shape mismatches or NaN errors.
- Benchmark: It establishes a minimum performance floor (e.g., MAE and $R^2$ score). Tomorrow (Day 67), when you build a more complex model, you will know for a fact whether the added complexity actually improved performance.Because demand_units is a continuous numerical variable, Linear Regression is the ideal baseline algorithm.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load the preprocessed splits saved from Day 65
train_df = pd.read_csv("retail_demand_train_processed.csv")
test_df = pd.read_csv("retail_demand_test_processed.csv")

X_train = train_df.drop(columns=['demand_units'])
y_train = train_df['demand_units']

X_test = test_df.drop(columns=['demand_units'])
y_test = test_df['demand_units']

# 2. Initialize and train the Baseline Linear Regression model
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

# 3. Generate predictions on the unseen test set
y_pred = baseline_model.predict(X_test)

# 4. Compute baseline evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("--- Day 66: Baseline Linear Regression Results ---")
print(f"Baseline MAE:  {mae:.2f} units")
print(f"Baseline RMSE: {rmse:.2f} units")
print(f"Baseline R²:   {r2:.4f}\n")

# 5. Save test set predictions to CSV for comparison on Day 67 & Day 68
results_df = pd.DataFrame({
    'actual_demand': y_test,
    'baseline_pred': np.round(y_pred, 2)
})
results_df.to_csv("retail_demand_predictions.csv", index=False)
print("Saved baseline predictions to 'retail_demand_predictions.csv'.")

--- Day 66: Baseline Linear Regression Results ---
Baseline MAE:  23.85 units
Baseline RMSE: 30.77 units
Baseline R²:   0.8313

Saved baseline predictions to 'retail_demand_predictions.csv'.


### Why Linear Regression Will Hit a Ceiling Here

Remember how we generated demand_units on Day 64? We multiplied base_demand by season_effect:$$\text{Demand} = (\text{Base Demand}) \times (\text{Season Multiplier})$$Linear regression is purely additive—it adds weighted features together ($\beta_1x_1 + \beta_2x_2$). It cannot naturally capture multiplicative interactions (such as how Festive season amplifies the effect of historical_sales_lag). This gives us a clear hypothesis for Day 67, where we will introduce non-linear tree-based models to break past this baseline ceiling.

### Guided Question:

Why did we re-attach `demand_units` to `train_df` and `test_df` before saving them to CSV, rather than saving `X_train`, `y_train`, X_t`est, and `y_test` into four separate CSV files?

### Answer:

Re-attaching the target variable (`demand_units`) to the feature matrices to create unified training (`train_df`) and test (`test_df`) datasets before saving them to CSV is a deliberate architectural choice centered on data integrity, alignment simplicity, and pipeline maintainability.

Saving them as a single cohesive table rather than four separate files (`X_train, y_train, X_test, y_test`) provides several key engineering advantages:

**1. Guaranteed Row Alignment (Index Integrity)**

When features and targets are stored in separate CSV files, you introduce a vulnerability: row misalignment.
- If a file gets accidentally sorted, truncated, or re-indexed during inspection or subsequent preprocessing steps, the strict one-to-one mapping between your feature rows (X) and target rows (y) can break silently.
- By keeping them in a single DataFrame (train_df), every feature row is permanently locked to its exact corresponding target value. You cannot accidentally mismatch features and labels when reloading them from disk.

**2. Reduced I/O Complexity and File Sprawl**

Managing fewer files simplifies your project structure and reduces boilerplate code:
- Four-file approach: Requires managing four distinct paths, four separate `pd.read_csv()` calls, and extra validation logic to ensure all shapes match.
- Two-file approach: Cleanly splits your entire ecosystem into two immutable, self-contained states: train_df (everything the model is allowed to see during training) and test_df (the strictly isolated evaluation set). Loading them requires just a single read followed by a clean drop:

`X_train = train_df.drop(columns=['demand_units'])
y_train = train_df['demand_units']`

**3. Safer Downstream Manipulations**
During iterative model development, you often need to filter rows (e.g., removing outliers, handling missing values, or filtering by specific date ranges or locations) after splitting.
- If features and targets are in separate files, any row-filtering operation must be manually synchronized across both files.
- When combined into a single DataFrame, any row filtering or slicing operation automatically applies to both the features and the target simultaneously, preventing shape-mismatch errors during model training (`ValueError: Found input variables with inconsistent numbers of samples`).

### Guided Question:

If your baseline Linear Regression model achieves an $R^2$ score of 0.85 in 0.01 seconds, and a complex model takes 10 minutes to train for an $R^2$ of 0.855, how does having the baseline inform your engineering decision?

### Answer:

This scenario presents a classic textbook example of diminishing returns in machine learning engineering. Having a fast, high-performing baseline immediately informs your decision to reject the complex model and deploy (or iterate further upon) the baseline.

**1. The ROI Asymmetry (Quantifying the Trade-off)**

When you run the numbers, the cost-benefit ratio heavily penalizes the complex model:
- Compute Cost: Training time increases from $0.01$ seconds to $600$ seconds—a $60,000\times$ increase in compute resources for model training.
- Performance Gain: The $R^2$ score improves by a negligible $0.005$ (from $0.85$ to $0.855$).

From an engineering ROI perspective, a $0.5\%$ improvement in explanatory power does not justify multiplying your training overhead, resource allocation, and carbon footprint by tens of thousands.

**2. Strategic Insights Provided by the Baseline**

A robust baseline transforms your development workflow by providing four critical operational guardrails:

**Fast Iteration Velocity ($0.01$s vs. $10$ mins)**

A model that trains in $0.01$ seconds enables instant feedback loops. You can run rapid feature engineering experiments, automated hyperparameter sweeps, cross-validation folds, and unit tests in a fraction of a second. A 10-minute training loop introduces friction, slowing down experimentation and making continuous integration (CI) pipelines sluggish.

**Operational Simplicity and Debuggability**

Linear regression coefficients have direct, transparent geometric and statistical interpretations. If predictions drift or behave unexpectedly in production, debugging a linear model takes minutes. A complex model (such as a deep neural network or heavily tuned gradient boosted machine) introduces a black-box layer that complicates traceability, compliance, and error auditing.

**Retraining and CI/CD Agility**

In production environments (such as retail demand prediction systems), data distributions shift over time, necessitating frequent model retraining on fresh data. A model that trains instantaneously can be retrained on the fly or scheduled continuously with zero infrastructure stress. A slow, heavy model requires dedicated GPU/CPU cluster orchestration and expensive pipeline scheduling.

**Mitigation of Overfitting and Variance**

Achieving a near-identical $R^2$ with massive model complexity often signals that the complex model is beginning to memorize noise in the training set rather than learning true underlying signals. The simpler baseline model provides better generalization security through structural regularization.

**Summary Recommendation**

Your baseline establishes that $85\%$ of the variance in the target variable is captured by simple linear relationships among your features.Before ever considering a complex architecture, your engineering roadmap should first exhaust simpler, low-cost modifications:
- Feature Engineering: Inject domain-specific interactions, rolling averages, or richer temporal indicators into the linear model to push 
R 
2
  higher without increasing model complexity.
- Regularization Tuning: Explore Ridge or Lasso variants (RidgeCV, LassoCV) to optimize coefficient shrinkage.

If the business domain strictly requires squeezing out every last decimal point of accuracy and the financial value of that 
0.5% improvement outweighs the infrastructure costs, only then should complexity be introduced. Otherwise, the baseline wins decisively.

### Guided Question:

Looking at how our synthetic retail demand was generated (multiplicative seasonal effects), why do we expect a standard Linear Regression model to underfit certain peak-season predictions?

### Answer:

A standard Linear Regression model is expected to underfit peak-season predictions because of a fundamental structural mismatch: 

Linear regression assumes an additive relationship between features, whereas multiplicative seasonal data generates effects that scale proportionally with the base magnitude of demand.

**1. The Core Mathematical Mismatch**

In a synthetic data-generating process with multiplicative seasonality, demand ($Y$) is modeled as a product of components:$$Y = \text{Trend} \times \text{Seasonality} \times \text{Price Effect} \times \epsilon$$

In contrast, a standard Linear Regression model assumes a strictly additive combination of coefficients and features:$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \epsilon$$

When you one-hot encode categorical seasonal features (like season = peak), linear regression treats the season as a constant additive shift (e.g., adding a flat $+500$ units). However, in a multiplicative system, a peak season does not add a fixed number of units—it multiplies the baseline demand by a factor (e.g., $1.5\times$ or $2.0\times$).

**2. Why This Causes Peak-Season Underfitting**

The Scale-Dependent FailureBecause the additive seasonal coefficient in a linear model is a global constant learned across the entire dataset, it computes an "averaged" compromise effect:
- For a low-volume product or location with a baseline demand of 200 units, a $2\times$ multiplicative peak means $+200$ units. A flat additive coefficient of $+500$ will overpredict demand.
- For a high-volume product or location with a baseline demand of 10,000 units, a $2\times$ multiplicative peak means $+10,000$ units. That same flat additive coefficient of $+500$ will severely underfit, missing the peak by thousands of units.

Because high-volume items heavily influence the least-squares optimization loss function, the linear model settles for an intermediate additive bump that drastically underestimates peak surges for high-throughput categories while failing to capture the true proportional swing.

Inability to Capture Feature InteractionsMultiplicative relationships require interactive terms (where features multiply each other, such as $\text{Base Demand} \times \text{Seasonality}$). A vanilla Linear Regression model only evaluates features independently unless you explicitly engineer interaction terms or non-linear transformations.

**3. How to Bridge the Gap**

To resolve this limitation without immediately jumping to a heavy black-box model, engineering teams typically employ two standard fixes:
- Log Transformation: Take the natural logarithm of the target variable ($\ln(Y)$). Applying a log transform turns a multiplicative relationship into an additive one:$$\ln(Y) = \ln(\text{Trend}) + \ln(\text{Seasonality}) + \dots$$

A standard linear model can then easily fit the additive log-values, and exponentiating the predictions restores the multiplicative scale.
- Explicit Interaction Features: Manually construct interaction terms (e.g., multiplying historical sales lag by seasonal indicators) to give the linear model the non-linear flexibility it needs to scale predictions during peak periods.